# ArrayExpress — Functional Genomics Data Ingestion and Analysis

**ArrayExpress** is EMBL-EBI's repository of functional genomics experiments, covering microarray and high-throughput sequencing (RNA-seq, ChIP-seq, ATAC-seq, and more). It has been integrated into the **BioStudies** infrastructure but retains its own search interface and accession namespace (`E-*`).

Key data types:
| Data type | Description |
|---|---|
| **Experiments** | Submissions with accessions like `E-MTAB-2836`; include metadata on organism, technology, and sample design |
| **Samples** | Individual biological samples with annotation (tissue, disease, treatment, etc.) |
| **Raw data files** | FASTQ / CEL / idat files linked from ENA or the BioStudies FTP |
| **Processed data** | Normalised expression matrices, peak call files, etc. |
| **IDF / SDRF** | MAGE-TAB format: Investigation Description File and Sample–Data Relationship File |

**Reference:** Athar et al. (2019), *Nucleic Acids Research*, ArrayExpress update — from bulk to single-cell expression data

**API base:** `https://www.ebi.ac.uk/biostudies/api/v1`

# TODO

* [x] **Ingest data**
    * [x] Connect to BioStudies/ArrayExpress API and confirm access (fetch one experiment, print title/organism/type)
    * [x] Page through experiments and download metadata (cache to `data/arrayexpress_experiments.json`)
    * [x] Parse into a Polars DataFrame: accession, title, organism, experiment_type, submission_date, release_date, technology, sample_count
    * [x] Fetch file listing for one well-known RNA-seq experiment (E-MTAB-2836)
    * [x] Summarise the DataFrame (shape, dtypes, head)
* [ ] **Explore and clean**
    * [ ] Summarise experiment counts by organism, technology, and year
    * [ ] Inspect missing values and normalise free-text fields
    * [ ] Plot submission timeline — how has the repository grown over time?
* [ ] **Analysis**
    * [ ] Identify most-studied organisms, tissues, and diseases
    * [ ] Compare microarray vs. sequencing experiment volumes over time
    * [ ] Cluster experiments by keyword / organism profile
* [ ] **Visualization**
    * [ ] Bar chart of top organisms and experiment types
    * [ ] Timeline of submissions by technology
    * [ ] Heatmap of organism × technology usage
* [ ] **Statistical analysis**
    * [ ] Explain the statistical considerations when comparing expression across platforms (microarray vs. RNA-seq)
    * [ ] Discuss batch-effect correction and multiple hypothesis testing in large-scale expression studies

In [ ]:
import json
import time
from pathlib import Path

import requests
import polars as pl

## 1. Ingest Data

### 1.1 Connect to BioStudies/ArrayExpress API and Confirm Access

In [ ]:
BIOSTUDIES_BASE = "https://www.ebi.ac.uk/biostudies/api/v1"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def biostudies_get(endpoint: str, params: dict = None, timeout: int = 30) -> dict:
    """Send a GET request to the BioStudies REST API and return parsed JSON.

    Parameters
    ----------
    endpoint : str
        API path relative to BIOSTUDIES_BASE (e.g. "studies/E-MTAB-2836").
    params : dict, optional
        Query parameters to append to the URL.
    timeout : int
        Request timeout in seconds.

    Returns
    -------
    dict
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a 4xx or 5xx status code.
    """
    url = f"{BIOSTUDIES_BASE}/{endpoint}"
    resp = requests.get(url, params=params or {}, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


# Connectivity check: fetch a single well-known experiment
SAMPLE_ACCESSION = "E-MTAB-2836"  # landmark human RNA-seq dataset
study = biostudies_get(f"studies/{SAMPLE_ACCESSION}")

# The BioStudies response wraps metadata in a list of 'section' attributes
# We extract the flat attribute list for quick display
attrs = {a["name"]: a["value"] for a in study.get("attributes", []) if "name" in a and "value" in a}

print(f"Accession       : {study.get('accno')}")
print(f"Title           : {attrs.get('Title', 'N/A')}")
print(f"Release date    : {study.get('releaseDate', attrs.get('Release Date', 'N/A'))}")
print(f"Organism        : {attrs.get('Organism', 'N/A')}")
print(f"Experiment type : {attrs.get('Experiment Type', 'N/A')}")

### 1.2 Page Through Experiments and Download Metadata

In [ ]:
EXPERIMENTS_CACHE = DATA_DIR / "arrayexpress_experiments.json"
PAGE_SIZE = 100  # maximum page size supported by the BioStudies search API


def fetch_all_experiments(cache_path: Path = EXPERIMENTS_CACHE) -> list[dict]:
    """Fetch ArrayExpress experiment metadata via the BioStudies search API.

    Paginates through the ``type=experiment`` search until all records are
    retrieved. Results are cached to *cache_path* as newline-delimited JSON
    so subsequent runs are instant.

    Parameters
    ----------
    cache_path : Path
        File path for the local JSON cache.

    Returns
    -------
    list[dict]
        One dict per experiment containing all fields returned by the API.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_hits: list[dict] = []
    page = 1  # BioStudies search uses 1-based pagination

    while True:
        resp = biostudies_get(
            "search",
            params={
                "type": "experiment",  # restrict to ArrayExpress experiments
                "page": page,
                "pageSize": PAGE_SIZE,
            },
            timeout=60,
        )

        hits = resp.get("hits", [])
        if not hits:
            break  # no more results

        all_hits.extend(hits)
        total = resp.get("totalHits", "?")
        print(f"  Page {page:>4} — {len(all_hits):>6} / {total}", end="\r")

        # Stop early if we have collected everything
        if isinstance(total, int) and len(all_hits) >= total:
            break

        page += 1
        time.sleep(0.3)  # polite delay between paginated requests

    print(f"\nDone. Fetched {len(all_hits)} experiments.")
    cache_path.write_text(json.dumps(all_hits))  # persist to disk
    return all_hits


experiments_raw = fetch_all_experiments()
print(f"Total cached experiments: {len(experiments_raw)}")

### 1.3 Parse Into a Polars DataFrame

In [ ]:
def flatten_experiment(hit: dict) -> dict:
    """Flatten a single raw BioStudies search hit into a row-friendly dict.

    The BioStudies search response stores most metadata inside a nested
    ``attributes`` list of ``{name, value}`` pairs. This function hoists
    the fields we care about to the top level.

    Parameters
    ----------
    hit : dict
        A single element from the ``hits`` list returned by the search API.

    Returns
    -------
    dict
        Flat dict suitable for building a Polars DataFrame row.
    """
    # Build a name → value lookup from the attribute list
    attrs: dict[str, str] = {}
    for attr in hit.get("attributes", []):
        name = attr.get("name", "")
        value = attr.get("value", "")
        if name and value and name not in attrs:  # keep first occurrence
            attrs[name] = str(value)

    # Sample count may be stored as an integer field or as a string attribute
    raw_samples = hit.get("numberOfComponents") or attrs.get("Number of samples")
    try:
        sample_count = int(raw_samples) if raw_samples is not None else None
    except (ValueError, TypeError):
        sample_count = None

    return {
        "accession":        hit.get("accession"),
        "title":            attrs.get("Title") or hit.get("title"),
        "organism":         attrs.get("Organism"),
        "experiment_type":  attrs.get("Experiment Type"),
        "technology":       attrs.get("Technology Type") or attrs.get("Technology"),
        "submission_date":  attrs.get("Submission Date") or hit.get("submissionDate"),
        "release_date":     attrs.get("Release Date") or hit.get("releaseDate"),
        "sample_count":     sample_count,
    }


# Build flat rows and load into Polars
rows = [flatten_experiment(h) for h in experiments_raw]
experiments = pl.DataFrame(rows).with_columns([
    # Parse ISO-8601 date strings; coerce unparseable values to null
    pl.col("submission_date").str.to_date("%Y-%m-%d", strict=False),
    pl.col("release_date").str.to_date("%Y-%m-%d", strict=False),
    pl.col("sample_count").cast(pl.Int32, strict=False),
])

print(f"Shape  : {experiments.shape}")
print(f"Memory : {experiments.estimated_size('mb'):.2f} MB")
print()
print(experiments.dtypes)  # confirm column types
experiments.head(5)

### 1.4 Fetch File Listing for E-MTAB-2836

In [ ]:
# E-MTAB-2836: Stegle et al. (2015) — single-cell RNA-seq of human iPSC lines
# The BioStudies files endpoint returns a JSON manifest of all associated files
FILES_ACCESSION = "E-MTAB-2836"
FILES_URL = f"https://www.ebi.ac.uk/biostudies/files/{FILES_ACCESSION}/{FILES_ACCESSION}.json"


def fetch_study_files(accession: str) -> pl.DataFrame:
    """Fetch the file manifest for an ArrayExpress/BioStudies study.

    Parameters
    ----------
    accession : str
        Study accession, e.g. ``"E-MTAB-2836"``.

    Returns
    -------
    pl.DataFrame
        One row per file with columns: ``path``, ``name``, ``size``, ``type``.
    """
    url = f"https://www.ebi.ac.uk/biostudies/files/{accession}/{accession}.json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    manifest = resp.json()

    # The manifest is a nested tree; flatten all leaf file nodes
    def extract_files(node: dict | list, parent_path: str = "") -> list[dict]:
        """Recursively extract file entries from the BioStudies file tree.

        Parameters
        ----------
        node : dict or list
            Current node in the file tree.
        parent_path : str
            Accumulated directory path from parent nodes.

        Returns
        -------
        list[dict]
            Flat list of file-level dicts.
        """
        results = []
        if isinstance(node, list):
            for item in node:
                results.extend(extract_files(item, parent_path))
        elif isinstance(node, dict):
            name = node.get("name", "")
            current_path = f"{parent_path}/{name}".lstrip("/")
            if "files" in node:  # directory node — recurse
                results.extend(extract_files(node["files"], current_path))
            else:  # leaf node — it's a file
                results.append({
                    "path":  current_path,
                    "name":  name,
                    "size":  node.get("size"),
                    "type":  node.get("type") or name.rsplit(".", 1)[-1] if "." in name else None,
                })
        return results

    file_list = extract_files(manifest)
    return pl.DataFrame(file_list).with_columns(
        pl.col("size").cast(pl.Int64, strict=False)
    )


study_files = fetch_study_files(FILES_ACCESSION)
print(f"Files in {FILES_ACCESSION}: {len(study_files)}")
print()

# Summarise by file extension / type
print(
    study_files
    .group_by("type")
    .agg(
        pl.len().alias("count"),
        (pl.col("size").sum() / 1e6).round(1).alias("total_mb"),
    )
    .sort("count", descending=True)
)
print()
study_files.head(10)

### 1.5 DataFrame Summary

In [ ]:
# ── Shape and dtypes ──────────────────────────────────────────────────────────
print(f"Shape  : {experiments.shape[0]:,} rows × {experiments.shape[1]} columns")
print(f"Memory : {experiments.estimated_size('mb'):.2f} MB")
print()
print("Column dtypes:")
for col, dtype in zip(experiments.columns, experiments.dtypes):
    print(f"  {col:<20} {dtype}")

# ── Null counts ───────────────────────────────────────────────────────────────
print("\nNull counts per column:")
print(experiments.null_count())

# ── Descriptive statistics ────────────────────────────────────────────────────
print("\nDescriptive statistics (numeric/date columns):")
print(experiments.describe())

# ── First few rows ────────────────────────────────────────────────────────────
print("\nHead (5 rows):")
experiments.head(5)